# 🚬 흡연 분류 AI 해커톤 - V2 (개선 버전)

**목표:** ROC-AUC 0.77+ 달성

**개선 사항:**
- 피처 엔지니어링 대폭 강화
- 하이퍼파라미터 세밀 튜닝
- 멀티시드 앙상블
- 가중치 최적화

---

## 📌 STEP 1: 환경 설정

In [ ]:
# 1-1. 라이브러리 설치
!pip install -q xgboost lightgbm catboost

In [ ]:
# 1-2. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1-3. 경로 설정 (본인 경로에 맞게 수정하세요)
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

In [ ]:
# 1-4. 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 머신러닝
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

# 부스팅 모델
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 시드 고정
import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)

print("✅ 라이브러리 임포트 완료!")

In [ ]:
# 1-5. 한글 폰트 설정 (Colab)
!apt-get update -qq
!apt-get install -qq fonts-nanum*

fe = fm.FontEntry(
    fname="/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    name="NanumGothic"
)
fm.fontManager.ttflist.insert(0, fe)
plt.rcParams.update({"font.size": 10, "font.family": "NanumGothic"})
plt.rcParams["axes.unicode_minus"] = False

print("✅ 한글 폰트 설정 완료!")

## 📌 STEP 2: 데이터 로드

In [ ]:
# 2-1. 데이터 로드
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print("=" * 50)
print("📊 데이터 기본 정보")
print("=" * 50)
print(f"Train: {train.shape}")
print(f"Test: {test.shape}")
print(f"\n컬럼: {train.columns.tolist()}")

In [ ]:
# 2-2. 기본 정보 확인
print("📋 데이터 미리보기:")
display(train.head())

print("\n📊 기술 통계량:")
display(train.describe())

print(f"\n🎯 타겟 분포:")
print(train['label'].value_counts())
print(f"흡연자 비율: {train['label'].mean()*100:.2f}%")

## 📌 STEP 3: 데이터 전처리

In [ ]:
# 3-1. 원본 복사
train_df = train.copy()
test_df = test.copy()

# 3-2. ID 분리 및 제거
test_id = test_df['ID'].copy() if 'ID' in test_df.columns else test_df['id'].copy() if 'id' in test_df.columns else None
train_df = train_df.drop(['ID', 'id'], axis=1, errors='ignore')
test_df = test_df.drop(['ID', 'id'], axis=1, errors='ignore')

# 3-3. 특성과 타겟 분리
X = train_df.drop('label', axis=1, errors='ignore')
y = train_df['label']
X_test = test_df.drop('label', axis=1, errors='ignore')

# 3-4. 컬럼명 확인 및 저장
feature_cols = X.columns.tolist()
print(f"특성 수: {len(feature_cols)}")
print(f"특성: {feature_cols}")

## 📌 STEP 4: 피처 엔지니어링 (대폭 강화!) ⭐

In [ ]:
def create_features_v2(df, feature_cols):
    """
    개선된 피처 엔지니어링 V2
    - 의학적으로 의미 있는 비율
    - 상호작용 특성
    - 구간화
    - 다항식 특성
    """
    df = df.copy()
    
    # 컬럼명 소문자로 통일 (대소문자 혼용 방지)
    col_map = {col: col.lower() for col in df.columns}
    df_lower = df.rename(columns=col_map)
    cols = df_lower.columns.tolist()
    
    # ========================================
    # 1. 콜레스테롤 관련 비율 (매우 중요!)
    # ========================================
    if 'hdl' in cols and 'ldl' in cols:
        df['HDL_LDL_ratio'] = df_lower['hdl'] / (df_lower['ldl'] + 1)
        df['LDL_HDL_ratio'] = df_lower['ldl'] / (df_lower['hdl'] + 1)
    
    if 'hdl' in cols and 'cholesterol' in cols:
        df['HDL_Chol_ratio'] = df_lower['hdl'] / (df_lower['cholesterol'] + 1)
        # 동맥경화 지수 (높을수록 위험)
        df['Atherogenic_index'] = (df_lower['cholesterol'] - df_lower['hdl']) / (df_lower['hdl'] + 1)
    
    if 'triglyceride' in cols and 'hdl' in cols:
        # TG/HDL 비율 (인슐린 저항성 지표, 흡연자가 높음)
        df['TG_HDL_ratio'] = df_lower['triglyceride'] / (df_lower['hdl'] + 1)
    
    # ========================================
    # 2. 혈압 관련
    # ========================================
    if 'systolic' in cols and 'diastolic' in cols:
        df['Pulse_pressure'] = df_lower['systolic'] - df_lower['diastolic']  # 맥압
        df['MAP'] = df_lower['diastolic'] + (df_lower['systolic'] - df_lower['diastolic']) / 3  # 평균동맥압
        df['BP_ratio'] = df_lower['systolic'] / (df_lower['diastolic'] + 1)
    
    # ========================================
    # 3. 간 기능 관련 (흡연과 연관!)
    # ========================================
    # 감마GTP - 흡연자가 높음
    if 'gtp' in cols:
        df['GTP_log'] = np.log1p(df_lower['gtp'])
        df['GTP_sqrt'] = np.sqrt(df_lower['gtp'])
        df['GTP_sq'] = df_lower['gtp'] ** 2
    
    # AST/ALT 비율 (드리티스 비율)
    if 'ast' in cols and 'alt' in cols:
        df['AST_ALT_ratio'] = df_lower['ast'] / (df_lower['alt'] + 1)
        df['Liver_index'] = df_lower['ast'] + df_lower['alt']
    
    # ========================================
    # 4. 헤모글로빈 관련 (흡연자가 높음!)
    # ========================================
    if 'hemoglobin' in cols:
        df['Hemo_sq'] = df_lower['hemoglobin'] ** 2
        df['Hemo_log'] = np.log1p(df_lower['hemoglobin'])
    
    # ========================================
    # 5. 신체 관련
    # ========================================
    if 'height' in cols and 'weight' in cols:
        # BMI 직접 계산
        df['BMI_calc'] = df_lower['weight'] / ((df_lower['height']/100) ** 2 + 0.01)
    
    if 'waist' in cols and 'height' in cols:
        df['Waist_Height_ratio'] = df_lower['waist'] / (df_lower['height'] + 1)
    
    # ========================================
    # 6. 시력/청력 관련
    # ========================================
    # 시력
    eyesight_cols = [c for c in cols if 'eyesight' in c]
    if len(eyesight_cols) >= 2:
        df['Eyesight_avg'] = df_lower[eyesight_cols].mean(axis=1)
        df['Eyesight_diff'] = abs(df_lower[eyesight_cols[0]] - df_lower[eyesight_cols[1]])
    
    # 청력
    hearing_cols = [c for c in cols if 'hearing' in c]
    if len(hearing_cols) >= 2:
        df['Hearing_sum'] = df_lower[hearing_cols].sum(axis=1)
    
    # ========================================
    # 7. 나이 상호작용 (누적 효과)
    # ========================================
    if 'age' in cols:
        age = df_lower['age']
        
        if 'hemoglobin' in cols:
            df['Age_x_Hemo'] = age * df_lower['hemoglobin']
        if 'gtp' in cols:
            df['Age_x_GTP'] = age * df_lower['gtp']
        if 'triglyceride' in cols:
            df['Age_x_TG'] = age * df_lower['triglyceride']
        if 'hdl' in cols:
            df['Age_x_HDL'] = age * df_lower['hdl']
        if 'systolic' in cols:
            df['Age_x_SBP'] = age * df_lower['systolic']
        
        # 나이 구간화
        df['Age_group'] = pd.cut(age, bins=[0, 30, 40, 50, 60, 100], labels=[0,1,2,3,4]).astype(float)
        df['Age_sq'] = age ** 2
    
    # ========================================
    # 8. 혈당 관련
    # ========================================
    fbs_col = [c for c in cols if 'blood' in c and 'sugar' in c or 'fasting' in c]
    if len(fbs_col) > 0:
        fbs = df_lower[fbs_col[0]]
        df['FBS_log'] = np.log1p(fbs)
        df['FBS_group'] = pd.cut(fbs, bins=[0, 100, 126, 500], labels=[0,1,2]).astype(float)
    
    # ========================================
    # 9. 중성지방 관련
    # ========================================
    if 'triglyceride' in cols:
        df['TG_log'] = np.log1p(df_lower['triglyceride'])
        df['TG_sq'] = df_lower['triglyceride'] ** 2
    
    # ========================================
    # 10. 통계 특성
    # ========================================
    # 건강 지표들의 통계
    health_cols = [c for c in cols if c in ['systolic', 'diastolic', 'hemoglobin', 
                                             'triglyceride', 'cholesterol', 'hdl', 'ldl']]
    if len(health_cols) >= 3:
        df['Health_mean'] = df_lower[health_cols].mean(axis=1)
        df['Health_std'] = df_lower[health_cols].std(axis=1)
        df['Health_max'] = df_lower[health_cols].max(axis=1)
        df['Health_min'] = df_lower[health_cols].min(axis=1)
        df['Health_range'] = df['Health_max'] - df['Health_min']
    
    # 결측치 처리
    df = df.fillna(0)
    
    # 무한값 처리
    df = df.replace([np.inf, -np.inf], 0)
    
    return df

# 피처 엔지니어링 적용
print("피처 엔지니어링 적용 중...")
X_fe = create_features_v2(X, feature_cols)
X_test_fe = create_features_v2(X_test, feature_cols)

print(f"\n✅ 피처 엔지니어링 완료!")
print(f"원본 특성 수: {len(feature_cols)}")
print(f"새로운 특성 수: {X_fe.shape[1]}")
print(f"\n새로 추가된 특성:")
new_cols = [c for c in X_fe.columns if c not in feature_cols]
print(new_cols)

## 📌 STEP 5: 이상치 처리 및 스케일링

In [ ]:
# 5-1. 이상치 처리 (RobustScaler 사용 - 이상치에 강건)
def clip_outliers(df, multiplier=3.0):
    """IQR 기반 이상치 클리핑 (더 넓은 범위)"""
    df = df.copy()
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64']:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - multiplier * IQR
            upper = Q3 + multiplier * IQR
            df[col] = df[col].clip(lower=lower, upper=upper)
    return df

# 이상치 클리핑
X_clipped = clip_outliers(X_fe, multiplier=3.0)
X_test_clipped = clip_outliers(X_test_fe, multiplier=3.0)

print("✅ 이상치 클리핑 완료!")

In [ ]:
# 5-2. Train/Validation 분할
X_train, X_val, y_train, y_val = train_test_split(
    X_clipped, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape}")
print(f"Val: {X_val.shape}")
print(f"Test: {X_test_clipped.shape}")

In [ ]:
# 5-3. 스케일링 (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test_clipped)

# 전체 데이터용 (나중에 최종 학습)
scaler_full = StandardScaler()
X_full_scaled = scaler_full.fit_transform(X_clipped)
X_test_final = scaler_full.transform(X_test_clipped)

print("✅ 스케일링 완료!")

## 📌 STEP 6: 기본 모델 성능 확인

In [ ]:
# 6-1. 기본 모델 테스트
print("=" * 60)
print("📊 기본 모델 성능 비교")
print("=" * 60)

base_models = {
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, verbosity=0, use_label_encoder=False, eval_metric='auc'),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(n_estimators=100, random_state=42, verbose=0),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

base_results = {}
for name, model in base_models.items():
    model.fit(X_train_scaled, y_train)
    train_pred = model.predict_proba(X_train_scaled)[:, 1]
    val_pred = model.predict_proba(X_val_scaled)[:, 1]
    
    train_auc = roc_auc_score(y_train, train_pred)
    val_auc = roc_auc_score(y_val, val_pred)
    
    base_results[name] = {'train': train_auc, 'val': val_auc}
    print(f"{name:15s} | Train: {train_auc:.5f} | Val: {val_auc:.5f} | Gap: {train_auc-val_auc:.5f}")

print("\n" + "=" * 60)

## 📌 STEP 7: 하이퍼파라미터 튜닝 (세밀화)

In [ ]:
# 7-1. XGBoost 튜닝 (세밀한 범위)
print("🔧 XGBoost 튜닝 중...")

xgb_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'min_child_weight': [1, 3, 5, 7],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [1, 2, 5]
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, verbosity=0, use_label_encoder=False, eval_metric='auc'),
    xgb_params,
    n_iter=80,
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_full_scaled, y)
print(f"\nXGBoost 최고 점수: {xgb_search.best_score_:.5f}")
print(f"최적 파라미터: {xgb_search.best_params_}")

In [ ]:
# 7-2. LightGBM 튜닝
print("🔧 LightGBM 튜닝 중...")

lgb_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 5, 7, -1],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'num_leaves': [15, 31, 63],
    'min_child_samples': [10, 20, 30, 50],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0, 0.1, 0.5]
}

lgb_search = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    lgb_params,
    n_iter=80,
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

lgb_search.fit(X_full_scaled, y)
print(f"\nLightGBM 최고 점수: {lgb_search.best_score_:.5f}")
print(f"최적 파라미터: {lgb_search.best_params_}")

In [ ]:
# 7-3. CatBoost 튜닝
print("🔧 CatBoost 튜닝 중...")

cat_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [4, 5, 6, 7],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'l2_leaf_reg': [1, 3, 5, 7],
    'border_count': [32, 64, 128]
}

cat_search = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=0),
    cat_params,
    n_iter=50,
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

cat_search.fit(X_full_scaled, y)
print(f"\nCatBoost 최고 점수: {cat_search.best_score_:.5f}")
print(f"최적 파라미터: {cat_search.best_params_}")

In [ ]:
# 7-4. Random Forest 튜닝
print("🔧 Random Forest 튜닝 중...")

rf_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params,
    n_iter=50,
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_full_scaled, y)
print(f"\nRandom Forest 최고 점수: {rf_search.best_score_:.5f}")
print(f"최적 파라미터: {rf_search.best_params_}")

In [ ]:
# 7-5. 튜닝 결과 요약
print("\n" + "=" * 60)
print("📊 하이퍼파라미터 튜닝 결과 요약")
print("=" * 60)
print(f"XGBoost:       {xgb_search.best_score_:.5f}")
print(f"LightGBM:      {lgb_search.best_score_:.5f}")
print(f"CatBoost:      {cat_search.best_score_:.5f}")
print(f"Random Forest: {rf_search.best_score_:.5f}")

## 📌 STEP 8: 멀티시드 앙상블 (핵심!)

In [ ]:
# 8-1. 멀티시드 모델 학습
print("=" * 60)
print("🎯 멀티시드 앙상블 구축")
print("=" * 60)

seeds = [42, 123, 456, 789, 1004]

# 각 모델별 최적 파라미터
best_xgb_params = xgb_search.best_params_
best_lgb_params = lgb_search.best_params_
best_cat_params = cat_search.best_params_
best_rf_params = rf_search.best_params_

# 검증용 예측 저장
val_preds = {'xgb': [], 'lgb': [], 'cat': [], 'rf': []}
test_preds = {'xgb': [], 'lgb': [], 'cat': [], 'rf': []}

for seed in seeds:
    print(f"\nSeed {seed} 학습 중...")
    
    # XGBoost
    xgb_model = XGBClassifier(**best_xgb_params, random_state=seed, verbosity=0, use_label_encoder=False, eval_metric='auc')
    xgb_model.fit(X_train_scaled, y_train)
    val_preds['xgb'].append(xgb_model.predict_proba(X_val_scaled)[:, 1])
    
    # LightGBM
    lgb_model = LGBMClassifier(**best_lgb_params, random_state=seed, verbose=-1)
    lgb_model.fit(X_train_scaled, y_train)
    val_preds['lgb'].append(lgb_model.predict_proba(X_val_scaled)[:, 1])
    
    # CatBoost
    cat_model = CatBoostClassifier(**best_cat_params, random_state=seed, verbose=0)
    cat_model.fit(X_train_scaled, y_train)
    val_preds['cat'].append(cat_model.predict_proba(X_val_scaled)[:, 1])
    
    # Random Forest
    rf_model = RandomForestClassifier(**best_rf_params, random_state=seed, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)
    val_preds['rf'].append(rf_model.predict_proba(X_val_scaled)[:, 1])

# 각 모델의 평균 예측
pred_xgb = np.mean(val_preds['xgb'], axis=0)
pred_lgb = np.mean(val_preds['lgb'], axis=0)
pred_cat = np.mean(val_preds['cat'], axis=0)
pred_rf = np.mean(val_preds['rf'], axis=0)

print("\n✅ 멀티시드 학습 완료!")
print(f"XGBoost 평균 Val AUC: {roc_auc_score(y_val, pred_xgb):.5f}")
print(f"LightGBM 평균 Val AUC: {roc_auc_score(y_val, pred_lgb):.5f}")
print(f"CatBoost 평균 Val AUC: {roc_auc_score(y_val, pred_cat):.5f}")
print(f"RandomForest 평균 Val AUC: {roc_auc_score(y_val, pred_rf):.5f}")

## 📌 STEP 9: 가중치 최적화

In [ ]:
# 9-1. 최적 가중치 탐색 (세밀하게)
print("=" * 60)
print("🔍 최적 가중치 탐색")
print("=" * 60)

best_score = 0
best_weights = None

# 0.05 단위로 세밀하게 탐색
for w1 in np.arange(0.1, 0.6, 0.05):
    for w2 in np.arange(0.1, 0.6, 0.05):
        for w3 in np.arange(0.1, 0.6, 0.05):
            w4 = round(1 - w1 - w2 - w3, 2)
            if w4 >= 0.05:
                weighted_pred = w1*pred_xgb + w2*pred_lgb + w3*pred_cat + w4*pred_rf
                score = roc_auc_score(y_val, weighted_pred)
                if score > best_score:
                    best_score = score
                    best_weights = (w1, w2, w3, w4)

print(f"\n🏆 최적 가중치:")
print(f"   XGBoost:      {best_weights[0]:.2f}")
print(f"   LightGBM:     {best_weights[1]:.2f}")
print(f"   CatBoost:     {best_weights[2]:.2f}")
print(f"   RandomForest: {best_weights[3]:.2f}")
print(f"\n🎯 가중 앙상블 Val AUC: {best_score:.5f}")

In [ ]:
# 9-2. 단순 평균과 비교
simple_avg = (pred_xgb + pred_lgb + pred_cat + pred_rf) / 4
simple_score = roc_auc_score(y_val, simple_avg)

print(f"\n📊 앙상블 방법 비교:")
print(f"   단순 평균:    {simple_score:.5f}")
print(f"   가중 평균:    {best_score:.5f}")
print(f"   개선:         {best_score - simple_score:.5f}")

## 📌 STEP 10: 최종 예측 및 제출 파일 생성

In [ ]:
# 10-1. 전체 데이터로 최종 모델 학습
print("=" * 60)
print("📝 최종 모델 학습 및 예측")
print("=" * 60)

# 최종 예측 저장
final_preds = {'xgb': [], 'lgb': [], 'cat': [], 'rf': []}

for seed in seeds:
    print(f"Seed {seed} 최종 학습 중...")
    
    # XGBoost
    xgb_model = XGBClassifier(**best_xgb_params, random_state=seed, verbosity=0, use_label_encoder=False, eval_metric='auc')
    xgb_model.fit(X_full_scaled, y)
    final_preds['xgb'].append(xgb_model.predict_proba(X_test_final)[:, 1])
    
    # LightGBM
    lgb_model = LGBMClassifier(**best_lgb_params, random_state=seed, verbose=-1)
    lgb_model.fit(X_full_scaled, y)
    final_preds['lgb'].append(lgb_model.predict_proba(X_test_final)[:, 1])
    
    # CatBoost
    cat_model = CatBoostClassifier(**best_cat_params, random_state=seed, verbose=0)
    cat_model.fit(X_full_scaled, y)
    final_preds['cat'].append(cat_model.predict_proba(X_test_final)[:, 1])
    
    # Random Forest
    rf_model = RandomForestClassifier(**best_rf_params, random_state=seed, n_jobs=-1)
    rf_model.fit(X_full_scaled, y)
    final_preds['rf'].append(rf_model.predict_proba(X_test_final)[:, 1])

print("\n✅ 최종 학습 완료!")

In [ ]:
# 10-2. 최종 앙상블 예측
# 각 모델의 멀티시드 평균
test_xgb = np.mean(final_preds['xgb'], axis=0)
test_lgb = np.mean(final_preds['lgb'], axis=0)
test_cat = np.mean(final_preds['cat'], axis=0)
test_rf = np.mean(final_preds['rf'], axis=0)

# 가중 평균 적용
w1, w2, w3, w4 = best_weights
final_prediction = w1*test_xgb + w2*test_lgb + w3*test_cat + w4*test_rf

print(f"예측값 범위: {final_prediction.min():.4f} ~ {final_prediction.max():.4f}")
print(f"예측값 평균: {final_prediction.mean():.4f}")

In [ ]:
# 10-3. 제출 파일 생성
submission_df = submission.copy()
submission_df['label'] = final_prediction

# 저장
output_path = result_path + 'submission_v2_ensemble.csv'
submission_df.to_csv(output_path, index=False)

print(f"\n✅ 제출 파일 저장: {output_path}")
print(f"\n제출 파일 미리보기:")
display(submission_df.head(10))

In [ ]:
# 10-4. 제출 파일 검증
print("\n🔍 제출 파일 검증:")
print(f"   행 개수: {len(submission_df)} (기대: 3001)")
print(f"   컬럼: {submission_df.columns.tolist()}")
print(f"   예측값 범위: {submission_df['label'].min():.4f} ~ {submission_df['label'].max():.4f}")
print(f"   결측치: {submission_df['label'].isnull().sum()}")

# 검증 통과 확인
if len(submission_df) == 3001 and submission_df['label'].isnull().sum() == 0:
    print("\n✅ 검증 통과! 제출 가능합니다.")
else:
    print("\n⚠️ 검증 실패! 데이터를 확인하세요.")

## 📌 STEP 11: 파일 다운로드

In [ ]:
# 11-1. Colab에서 파일 다운로드
from google.colab import files

# 다운로드 실행
files.download(output_path)

print("\n📥 다운로드 완료!")
print("다운로드된 파일: submission_v2_ensemble.csv")
print("\n이 파일을 해커톤 사이트에 제출하세요!")

## 📌 STEP 12: Feature Importance 분석

In [ ]:
# 12-1. Feature Importance 시각화
# 마지막으로 학습된 XGBoost 모델 사용
feature_names = X_clipped.columns.tolist()

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

# Top 25 시각화
plt.figure(figsize=(12, 10))
plt.barh(importance_df.head(25)['feature'], importance_df.head(25)['importance'])
plt.xlabel('Importance')
plt.title('Top 25 Feature Importance (XGBoost)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\n📊 Top 15 중요 특성:")
print(importance_df.head(15).to_string(index=False))

## 📌 결과 요약

---

### 🎯 개선된 V2 버전의 특징

| 항목 | V1 | V2 |
|------|----|----|  
| 피처 수 | ~20개 | **40개+** |
| 튜닝 반복 | 50회 | **80회** |
| 앙상블 | 단일 시드 | **5개 시드 평균** |
| 가중치 탐색 | 0.1 단위 | **0.05 단위** |

### 📈 예상 점수 향상
- V1: ~0.765
- V2: **~0.77+**

---

In [ ]:
print("=" * 60)
print("🎉 모든 작업 완료!")
print("=" * 60)
print(f"\n📁 제출 파일: {output_path}")
print(f"🎯 예상 Validation AUC: {best_score:.5f}")
print("\n해커톤 사이트에서 제출 후 결과를 확인하세요!")
print("점수가 낮으면 피처 엔지니어링을 더 추가해보세요.")